# 💡 Business Insights Report — SwiftEats

Executive-ready insights with actionable recommendations.

## Setup

In [ ]:
import os, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sqlalchemy import create_engine, text
from datetime import datetime
from dotenv import load_dotenv
load_dotenv()
engine = create_engine(
    f"postgresql://{os.getenv('DB_USER','food_user')}:{os.getenv('DB_PASSWORD','')}"
    f"@{os.getenv('DB_HOST','localhost')}:{os.getenv('DB_PORT','5432')}"
    f"/{os.getenv('DB_NAME','food_delivery')}"
)
REPORT_DATE = datetime.now().strftime('%Y-%m-%d')
print(f"Report date: {REPORT_DATE}")

## 1. Monthly GMV with Growth

In [ ]:
with engine.connect() as conn:
    df = pd.read_sql(text('''
        SELECT day, gross_revenue AS gmv, platform_revenue,
               total_orders, active_customers, avg_order_value
        FROM mv_daily_kpi
        WHERE EXTRACT(DAY FROM day) = 1
        ORDER BY day DESC LIMIT 18
    '''), conn).sort_values('day')

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(df['day'], df['gmv'], alpha=0.2, color='steelblue')
ax.plot(df['day'], df['gmv'], 'o-', color='steelblue', label='GMV', linewidth=2)
ax.plot(df['day'], df['platform_revenue'], 'o-', color='green',
        label='Platform Revenue', linewidth=2)
ax.set_title('Monthly GMV vs Platform Revenue')
ax.set_ylabel('Revenue (₹)')
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

# Print summary
latest = df.iloc[-1]
prev   = df.iloc[-2]
print(f"Latest month GMV:    ₹{latest['gmv']:>10,.0f}")
print(f"Platform Revenue:    ₹{latest['platform_revenue']:>10,.0f}")
print(f"Take Rate:           {latest['platform_revenue']/latest['gmv']*100:.1f}%")
print(f"MoM GMV Growth:      {(latest['gmv']-prev['gmv'])/prev['gmv']*100:+.1f}%")

## 2. Key Business Recommendations

In [ ]:
with engine.connect() as conn:
    churn = pd.read_sql(text('''
        SELECT COUNT(*) FILTER(WHERE recency_days > 60 AND monetary > 2000) AS high_val_at_risk,
               ROUND(SUM(monetary) FILTER(WHERE recency_days > 60 AND monetary > 2000)::numeric,0) AS revenue_at_risk
        FROM mv_customer_rfm
    '''), conn).iloc[0]

    sla = pd.read_sql(text('''
        SELECT ROUND(AVG(CASE WHEN sla_breached THEN 1 ELSE 0 END)*100::numeric,1) AS sla_breach_pct
        FROM deliveries WHERE delivery_status = 'Delivered'
    '''), conn).iloc[0]

print("=" * 55)
print(f"  SWIFTEATS BUSINESS INSIGHTS — {REPORT_DATE}")
print("=" * 55)
print()
print("⚠️  CHURN ALERT")
print(f"  {int(churn['high_val_at_risk']):,} high-value customers (spent >₹2K)")
print(f"  haven't ordered in 60+ days.")
print(f"  Revenue at risk: ₹{float(churn['revenue_at_risk']):,.0f}")
print(f"  → Recommendation: Send ₹100 win-back voucher to this segment")
print()
print("🚚 DELIVERY SLA")
print(f"  Current SLA breach rate: {float(sla['sla_breach_pct']):.1f}%")
if float(sla['sla_breach_pct']) > 10:
    print("  → Action: Increase partner supply in high-breach zones")
    print("    Focus on dinner rush (19-22h) and weekend afternoons")
else:
    print("  ✅ SLA within target (<10%)")
print("=" * 55)